In [12]:
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

df = pd.read_parquet("../data/interim/accepted_2007_to_2018Q4.parquet")

df = df.drop(columns=["member_id", "id"])

resolved = df[df["loan_status"].isin(["Fully Paid", "Charged Off", "Default"])].copy()
resolved["target"] = (
    resolved["loan_status"].isin(["Charged Off", "Default"]).astype(int)
)

print(resolved["target"].mean())

X = resolved.select_dtypes(include="number").drop(columns=["target"])
y = resolved["target"]


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
model = HistGradientBoostingClassifier(random_state=42)

model.fit(X_train, y_train)


proba = model.predict_proba(X_test)[:, 1]

print(roc_auc_score(y_test, proba))


0.19964990522912254
0.9991092373996443


## Filtering

The same logic used in week 1 for excluding non-known `loan_status` outcomes. Added a `target` integer binary column for weather a record is defaulted or not.

## Test and Training Data

`X` is the numeric columns from the table excluding `target` column. While `y` only includes the target column `target`. These two are passed to `train_test_split`, which returns training and testing data. The test size is 20%. `train_test_split` is used so we don't test and train our model on the same data.

## Model

We are using `HistGradientBoostingClassifier`, which is a histogram-based gradient boosted decision tree model. The model is fit with the train data. Then it predicts the predicted probability on the test data. To the get AUC (Area Under the ROC Curve) we use `roc_auc_score`. The model scores ~99.91%, which is extremely high. This is because of leakage. Some columns are not known to a lender when the loan is agreed upon which makes the model score higher than it should.


In [17]:
column_names = X_test.columns
for i, col in enumerate(column_names):
    mask = X_test[col].notna()
    prob = roc_auc_score(y_test[mask], X_test.loc[mask, col])
    score = max(prob, 1 - prob)
    if score > 0.6:
        print(f"{i}: {col}")


3: int_rate
20: total_pymnt
21: total_pymnt_inv
22: total_rec_prncp
25: recoveries
26: collection_recovery_fee
27: last_pymnt_amnt
28: last_fico_range_high
29: last_fico_range_low
34: dti_joint
91: sec_app_fico_range_low
92: sec_app_fico_range_high
94: sec_app_mort_acc
101: sec_app_mths_since_last_major_derog
103: hardship_amount
105: hardship_dpd
107: hardship_payoff_balance_amount
109: settlement_amount
110: settlement_percentage
111: settlement_term


## Leakage Investigation
